# Neon Trade Tycoon – Notebook Edition

Interaktive Wirtschaftshandelssimulation für **Jupyter/IPython** mit `ipywidgets`.

**Features**
- Dynamische Marktpreise je Stadt
- Kaufen/Verkaufen mit Laderaum
- Reisen, Treibstoff, Schulden + Zinsen
- Bedienung über Dropdowns + Buttons


In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass, field
from typing import Dict, List

from IPython.display import HTML, display
import ipywidgets as widgets

GOODS = ["Getreide", "Eisen", "Gewürze", "Tuch", "Werkzeuge", "Luxusgüter"]
CITY_NAMES = ["Aurelia", "Dornhafen", "Eiswacht", "Kupferfurt", "Neonheim"]
BASE_PRICES = {
    "Getreide": 32,
    "Eisen": 75,
    "Gewürze": 140,
    "Tuch": 95,
    "Werkzeuge": 120,
    "Luxusgüter": 220,
}


@dataclass
class City:
    name: str
    prices: Dict[str, int] = field(default_factory=dict)


@dataclass
class GameState:
    money: int = 1200
    debt: int = 500
    day: int = 1
    cargo_capacity: int = 24
    fuel: int = 8
    max_fuel: int = 8
    location_index: int = 0
    status: str = "Willkommen im Markt!"
    cargo: Dict[str, int] = field(default_factory=lambda: {g: 0 for g in GOODS})
    cities: List[City] = field(default_factory=list)

    @property
    def location(self) -> City:
        return self.cities[self.location_index]

    @property
    def cargo_used(self) -> int:
        return sum(self.cargo.values())

    @property
    def net_worth(self) -> int:
        cargo_value = sum(self.cargo[g] * self.location.prices[g] for g in GOODS)
        return self.money + cargo_value - self.debt


class NotebookTradeSim:
    def __init__(self, seed: int | None = None) -> None:
        self.rng = random.Random(seed)
        self.state = GameState(cities=[City(name=n) for n in CITY_NAMES])
        self.generate_market(initial=True)

    def generate_market(self, initial: bool = False) -> None:
        for city in self.state.cities:
            prices: Dict[str, int] = {}
            for good in GOODS:
                base = BASE_PRICES[good]
                volatility = self.rng.uniform(0.6, 1.5)
                bonus = 1.0
                if city.name == "Eiswacht" and good in {"Eisen", "Werkzeuge"}:
                    bonus = 0.82
                elif city.name == "Dornhafen" and good in {"Getreide", "Tuch"}:
                    bonus = 0.84
                elif city.name == "Aurelia" and good in {"Luxusgüter", "Gewürze"}:
                    bonus = 0.79
                prices[good] = int(max(10, base * volatility * bonus))
            city.prices = prices

        if not initial:
            self.state.status = "Neuer Tag: Marktpreise haben sich geändert."

    def rank(self) -> str:
        wealth = self.state.net_worth
        if wealth > 7000:
            return "Magnat"
        if wealth > 4000:
            return "Aufsteiger"
        if wealth > 1800:
            return "Händler"
        return "Anfänger"

    def buy(self, good: str) -> None:
        st = self.state
        price = st.location.prices[good]
        if st.cargo_used >= st.cargo_capacity:
            st.status = "Laderaum voll."
            return
        if st.money < price:
            st.status = "Zu wenig Gold."
            return
        st.money -= price
        st.cargo[good] += 1
        st.status = f"Gekauft: 1x {good} für {price} G."

    def sell(self, good: str) -> None:
        st = self.state
        if st.cargo[good] <= 0:
            st.status = f"Kein {good} im Laderaum."
            return
        price = st.location.prices[good]
        st.cargo[good] -= 1
        st.money += price
        st.status = f"Verkauft: 1x {good} für {price} G."

    def travel(self, city_name: str) -> None:
        st = self.state
        target = CITY_NAMES.index(city_name)
        if target == st.location_index:
            st.status = "Du bist bereits dort."
            return
        if st.fuel <= 0:
            st.status = "Kein Treibstoff. Tanke zuerst."
            return
        st.location_index = target
        st.fuel -= 1
        self.advance_day("Reise abgeschlossen.")

    def refuel(self) -> None:
        st = self.state
        cost = 160
        if st.fuel >= st.max_fuel:
            st.status = "Tank ist schon voll."
            return
        if st.money < cost:
            st.status = "Nicht genug Gold zum Tanken."
            return
        st.money -= cost
        st.fuel = st.max_fuel
        st.status = "Tank aufgefüllt."

    def repay_debt(self) -> None:
        st = self.state
        if st.debt <= 0:
            st.status = "Du bist schuldenfrei."
            return
        amount = min(250, st.money, st.debt)
        if amount <= 0:
            st.status = "Kein Gold für Tilgung."
            return
        st.money -= amount
        st.debt -= amount
        st.status = f"{amount} G Schulden getilgt."

    def advance_day(self, msg: str = "Nächster Tag.") -> None:
        st = self.state
        st.day += 1
        st.debt = int(st.debt * 1.02)
        self.generate_market()
        st.status = msg


sim = NotebookTradeSim()


In [ ]:
good_dd = widgets.Dropdown(options=GOODS, description='Ware:')
city_dd = widgets.Dropdown(options=CITY_NAMES, description='Stadt:')

btn_buy = widgets.Button(description='Kaufen', button_style='success')
btn_sell = widgets.Button(description='Verkaufen', button_style='warning')
btn_travel = widgets.Button(description='Reisen', button_style='info')
btn_refuel = widgets.Button(description='Tanken')
btn_debt = widgets.Button(description='Schulden zahlen')
btn_next = widgets.Button(description='Nächster Tag')
btn_reset = widgets.Button(description='Neu starten', button_style='danger')

status_out = widgets.Output()
table_out = widgets.Output()


def render_html() -> str:
    st = sim.state
    market_rows = ''.join(
        f"<tr><td>{g}</td><td>{st.location.prices[g]} G</td><td>{st.cargo[g]}</td></tr>"
        for g in GOODS
    )
    city_rows = ''.join(
        f"<tr><td>{c.name}</td><td>{'Ja' if i == st.location_index else 'Nein'}</td></tr>"
        for i, c in enumerate(st.cities)
    )

    return f"""
    <div style='font-family:Inter,Segoe UI,Arial,sans-serif'>
      <h2 style='margin-bottom:0.2rem'>💹 Neon Trade Tycoon (Notebook)</h2>
      <p style='margin-top:0'>Tag <b>{st.day}</b> | Ort <b>{st.location.name}</b> | Gold <b>{st.money}</b> | Schulden <b>{st.debt}</b> | Treibstoff <b>{st.fuel}/{st.max_fuel}</b> | Laderaum <b>{st.cargo_used}/{st.cargo_capacity}</b></p>
      <p><b>Nettovermögen:</b> {st.net_worth} | <b>Rang:</b> {sim.rank()}</p>
      <div style='display:flex;gap:1.5rem;align-items:flex-start'>
        <div>
          <h4>Markt ({st.location.name})</h4>
          <table border='1' cellpadding='6' cellspacing='0'>
            <tr><th>Ware</th><th>Preis</th><th>Lager</th></tr>
            {market_rows}
          </table>
        </div>
        <div>
          <h4>Städte</h4>
          <table border='1' cellpadding='6' cellspacing='0'>
            <tr><th>Name</th><th>Aktuell?</th></tr>
            {city_rows}
          </table>
        </div>
      </div>
      <p style='margin-top:1rem'><b>Status:</b> {st.status}</p>
    </div>
    """


def refresh(*_):
    with table_out:
        table_out.clear_output(wait=True)
        display(HTML(render_html()))


def on_buy(_):
    sim.buy(good_dd.value)
    refresh()


def on_sell(_):
    sim.sell(good_dd.value)
    refresh()


def on_travel(_):
    sim.travel(city_dd.value)
    refresh()


def on_refuel(_):
    sim.refuel()
    refresh()


def on_debt(_):
    sim.repay_debt()
    refresh()


def on_next(_):
    sim.advance_day('Du wartest einen Tag und beobachtest den Markt.')
    refresh()


def on_reset(_):
    global sim
    sim = NotebookTradeSim()
    refresh()


btn_buy.on_click(on_buy)
btn_sell.on_click(on_sell)
btn_travel.on_click(on_travel)
btn_refuel.on_click(on_refuel)
btn_debt.on_click(on_debt)
btn_next.on_click(on_next)
btn_reset.on_click(on_reset)

controls_row1 = widgets.HBox([good_dd, btn_buy, btn_sell])
controls_row2 = widgets.HBox([city_dd, btn_travel, btn_refuel])
controls_row3 = widgets.HBox([btn_debt, btn_next, btn_reset])
ui = widgets.VBox([controls_row1, controls_row2, controls_row3, table_out])

refresh()
display(ui)
